In [152]:
import geopandas as gpd
import glob
import numpy as np
import pandas as pd

METERS_TO_ACRES = 4046.86
ACREFT_TO_METERS3 = 1233.48
MM_TO_IN = 25.4
IN_TO_FT= 12

In [153]:
masktypes = ['ag']#['all','ag'] # Need to finish county stitching for all lands

scales = ['county','gw_basin']#['hydrologic_region','gw_basin','county'] # Need hydrologic_region shapefile

id_columns = {'county':['NAME','county_id'],
             'gw_basin':['Basin_Subb','basin_id'],
             'hydrologic_region':['HR_NAME','hr_id']}

for scale in scales:
    print(scale)
    id_colname = id_columns[scale][0]
    cols_to_add = [id_colname, "area_acres", "max_mask_a"]
    rename_id = id_columns[scale][1]
    
    for masktype in masktypes:
        # if masktype =='all':
        csv_in_path = f'csv_{scale}_{masktype}_lands/{scale}_{masktype}_lands_all_models.csv'
        csv_data_df = pd.read_csv(csv_in_path)
        csv_data_df = csv_data_df.loc[csv_data_df.MODEL=='ENSEMBLE']

        gdf = gpd.read_file(f'shapefile_{scale}_{masktype}_lands/ensemble_mean.shp')
        all_data_df = csv_data_df.merge(gdf[cols_to_add].drop_duplicates(subset=id_colname),
                                      on=id_colname,
                                      how="left")
        all_data_df['ET_mean_in']= all_data_df['ET_MEAN']/MM_TO_IN
        all_data_df['ET_acre_ft']= (all_data_df['ET_mean_in']/IN_TO_FT)*all_data_df['max_mask_a']
        all_data_df["DATE"] = pd.to_datetime(all_data_df["DATE"])
        all_data_df["year_month"] = all_data_df["DATE"].dt.strftime("%Y-%m")
        all_data_df["year"] = all_data_df["DATE"].dt.strftime("%Y")
        all_data_df["month"] = all_data_df["DATE"].dt.strftime("%m")
        all_data_df['timestep']='month'
        print(scale, masktype, id_colname,rename_id)
        # print(df.columns)
        out_df = all_data_df[[id_colname,'MODEL','area_acres','max_mask_a','ET_mean_in','ET_acre_ft','year_month','year','month','timestep']].copy()

county
county ag NAME county_id
gw_basin
gw_basin ag Basin_Subb basin_id


In [151]:
out_df.MODEL.unique()


array(['DISALEXI', 'EEMETRIC', 'ENSEMBLE', 'GEESEBAL', 'PTJPL', 'SIMS',
       'SSEBOP'], dtype=object)

In [144]:
# Work from your monthly table
out_df["year"] = out_df["year"].astype(int)
out_df["month"] = out_df["month"].astype(int)

# Add water year to monthly rows
out_df["water_year"] = np.where(
    out_df["month"] >= 10,
    out_df["year"] + 1,
    out_df["year"]
)

group_cols = [
    "Basin_Subb",
    "MODEL",
    "area_acres",
    "max_mask_a",
    "water_year"
]

def sum_only_complete_water_year(group):
    required_months = set(range(1, 13))
    months_present = set(group["month"].dropna().astype(int))

    has_all_months = months_present == required_months
    has_no_missing = group[["ET_mean_in", "ET_acre_ft"]].notna().all().all()

    return pd.Series({
        "ET_mean_in": group["ET_mean_in"].sum() if has_all_months and has_no_missing else np.nan,
        "ET_acre_ft": group["ET_acre_ft"].sum() if has_all_months and has_no_missing else np.nan,
        "n_months": group["month"].nunique(),
        "n_missing_ET_mean_in": group["ET_mean_in"].isna().sum(),
        "n_missing_ET_acre_ft": group["ET_acre_ft"].isna().sum()
    })

# Create water-year total rows
water_year_rows = (
    out_df
    .groupby(group_cols, dropna=False)
    .apply(sum_only_complete_water_year)
    .reset_index()
)

# Assign row-level fields so columns match monthly table
water_year_rows["timestep"] = "water_year"
water_year_rows["year"] = np.nan

# Choose ONE of these month options:
water_year_rows["month"] = np.nan
# water_year_rows["month"] = 0

water_year_rows["year_month"] = water_year_rows["water_year"].astype(str)+'-00'

# Add any missing columns so concat preserves the full schema
for col in out_df.columns:
    if col not in water_year_rows.columns:
        water_year_rows[col] = np.nan

# Reorder water-year rows to match out_df columns
water_year_rows = water_year_rows[out_df.columns]

# Append water-year rows to the original monthly table
out_all_data_df = pd.concat( [out_df, water_year_rows],
    ignore_index=True)

out_all_data_df.to_csv(f'all_data_{scale}.csv')


### Headers for data summers for SGMA data viewer
#### basin_id: Basin_Subb
#### county_id: COUNTY_Nam
#### hr_id: HR_NAME

#### hr
hr_id, area_acres, max_mask_area_acres, mean_inches, mean_acre_ft, timestep, year, month, year_month
#### basin
basin_id, area_acres, max_mask_area_acres, mean_inches, mean_acre_ft, timestep, year, month, year_month
#### county
county_id, area_acres, max_mask_area_acres, mean_inches, mean_acre_ft, timestep, year, month, year_month